In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/credit_data_clean.csv")
print(df.shape)
df.head()

(1860331, 89)


,Unnamed: 0,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,debt_settlement_flag,target
0,0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,10+ years,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0
1,1,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,< 1 year,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,1
2,2,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,10+ years,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0
3,3,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,10+ years,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0
4,4,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,1 year,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,N,N,0


In [3]:
X = df.drop(columns=["target"])
y = df["target"]

In [4]:
df.columns

Index(['Unnamed: 0', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term',
       'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
       'loan_status', 'pymnt_plan', 'purpose', 'addr_state', 'dti',
       'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv',
       'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
       'last_fico_range_high', 'last_fico_range_low',
       'collections_12_mths_ex_med', 'policy_code', 'application_type',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util'

In [5]:
categorical_cols = [
    'term',                 # e.g., '36 months', '60 months'
    'grade',                # A–G
    'sub_grade',            # A1–G5
    'emp_length',           # '< 1 year', '10+ years'
    'home_ownership',       # RENT, MORTGAGE, OWN, etc.
    'verification_status',  # Not Verified, Verified
    'purpose',              # debt_consolidation, credit_card, etc.
    'addr_state',           # state codes
    'initial_list_status',  # 'w' or 'f'
    'application_type',     # Individual, Joint App
    'hardship_flag',        # Y/N
    'debt_settlement_flag', # Y/N
    'pymnt_plan'            # usually 'n' or 'y'
]


In [6]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["issue_year"] = df["issue_d"].dt.year
df["issue_month"] = df["issue_d"].dt.month

In [7]:
# df.drop(columns=["issue_d"], inplace=True)


In [8]:
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")
df["credit_age_months"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days // 30
df["credit_age_months"] = df["credit_age_months"].clip(lower=0)  # Ensure non-negative


In [9]:
df.drop(columns=["last_pymnt_d", "last_credit_pull_d","issue_d","earliest_cr_line"], inplace=True)
df

,Unnamed: 0,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,debt_settlement_flag,target,issue_year,issue_month,credit_age_months
0,0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,10+ years,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,327
1,1,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,< 1 year,...,NaN,NaN,NaN,NaN,N,N,1,2011,12,154
2,2,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,10+ years,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,122
3,3,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,10+ years,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,192
4,4,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,1 year,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860326,105446,24000.0,24000.0,24000.0,60 months,23.99%,690.30,E,E2,< 1 year,...,103322.0,60812.0,28200.0,64422.0,N,N,1,2017,4,267
1860327,105447,10000.0,10000.0,10000.0,36 months,7.99%,313.32,A,A5,10+ years,...,132303.0,55863.0,34800.0,70203.0,N,N,0,2017,4,287
1860328,105448,10050.0,10050.0,10050.0,36 months,16.99%,358.26,D,D1,8 years,...,30400.0,14300.0,9000.0,0.0,N,N,1,2017,4,291
1860329,105449,6000.0,6000.0,6000.0,36 months,11.44%,197.69,B,B4,5 years,...,47476.0,26201.0,8100.0,34076.0,N,N,0,2017,4,327


In [12]:
leakage_columns = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'last_fico_range_high',
    'last_fico_range_low', 'loan_status', 'Unnamed: 0'
]

df = df.drop(columns=leakage_columns)


In [13]:
df

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,debt_settlement_flag,target,issue_year,issue_month,credit_age_months
0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,10+ years,RENT,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,327
1,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,< 1 year,RENT,...,NaN,NaN,NaN,NaN,N,N,1,2011,12,154
2,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,10+ years,RENT,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,122
3,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,10+ years,RENT,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,192
4,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,1 year,RENT,...,NaN,NaN,NaN,NaN,N,N,0,2011,12,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860326,24000.0,24000.0,24000.0,60 months,23.99%,690.30,E,E2,< 1 year,RENT,...,103322.0,60812.0,28200.0,64422.0,N,N,1,2017,4,267
1860327,10000.0,10000.0,10000.0,36 months,7.99%,313.32,A,A5,10+ years,MORTGAGE,...,132303.0,55863.0,34800.0,70203.0,N,N,0,2017,4,287
1860328,10050.0,10050.0,10050.0,36 months,16.99%,358.26,D,D1,8 years,RENT,...,30400.0,14300.0,9000.0,0.0,N,N,1,2017,4,291
1860329,6000.0,6000.0,6000.0,36 months,11.44%,197.69,B,B4,5 years,RENT,...,47476.0,26201.0,8100.0,34076.0,N,N,0,2017,4,327


In [14]:
# Define known columns to exclude
excluded_cols = [
    'Unnamed: 0', 'loan_status', 'target',               # Not useful or already encoded
    'term', 'grade', 'sub_grade', 'emp_length',          # Categorical
    'home_ownership', 'verification_status', 'purpose',
    'addr_state', 'initial_list_status', 'application_type',
    'hardship_flag', 'debt_settlement_flag', 'pymnt_plan'
]

categorical_cols = [
    'term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
    'verification_status', 'purpose', 'addr_state', 'initial_list_status',
    'application_type', 'hardship_flag', 'debt_settlement_flag', 'pymnt_plan'
]

# Numerical columns = everything not in excluded
numerical_cols = [col for col in df.columns
                  if col not in excluded_cols and col not in categorical_cols]


In [15]:
print("Numerical columns (sample):", numerical_cols[:10])
print("Total numerical columns:", len(numerical_cols))


Numerical columns (sample): ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high']
Total numerical columns: 60


In [18]:
df.to_csv("clean1.csv")

In [16]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import joblib

In [17]:
X = df.drop(columns=["target"])
y = df["target"]

In [19]:
# Remove % sign and convert to float
X["int_rate"] = X["int_rate"].str.strip().str.replace("%", "").astype(float)

In [20]:
# Detect columns with percentage values (object dtype + contains '%')
percent_cols = [col for col in X.columns if X[col].dtype == 'object' and X[col].astype(str).str.contains('%').any()]

# Clean and convert them to float
for col in percent_cols:
    X[col] = X[col].astype(str).str.strip().str.replace('%', '', regex=False).astype(float)

print(f"Cleaned percentage columns: {percent_cols}")

Cleaned percentage columns: ['revol_util']


In [21]:
def clean_percent_column(series):
    return series.apply(lambda x: float(str(x).replace('%', '').strip()) if isinstance(x, str) and '%' in x else x)


X['revol_util'] = clean_percent_column(X['revol_util'])


In [22]:
print(X.dtypes[X.dtypes == 'object'])


term                    object
grade                   object
sub_grade               object
emp_length              object
home_ownership          object
verification_status     object
pymnt_plan              object
purpose                 object
addr_state              object
initial_list_status     object
application_type        object
hardship_flag           object
debt_settlement_flag    object
dtype: object


In [23]:
# Numerical pipeline: Impute with median, then scale
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline: Impute with mode, then one-hot encode
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into a full preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])


In [24]:
X_processed = preprocessor.fit_transform(X)
print("Processed shape:", X_processed.shape)

Processed shape: (1860331, 198)


In [43]:
X_processed

array([[-1.06887082, -1.06850681, -1.06916363, ...,  1.        ,
         0.        ,  1.        ],
       [-1.34756305, -1.34728134, -1.34510439, ...,  1.        ,
         0.        ,  1.        ],
       [-1.35871074, -1.35843232, -1.35625351, ...,  1.        ,
         0.        ,  1.        ],
       ...,
       [-0.50591252, -0.50538224, -0.50334571, ...,  1.        ,
         0.        ,  1.        ],
       [-0.95739393, -0.95699699, -0.95488513, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.71805147,  1.71923856,  1.72090404, ...,  1.        ,
         0.        ,  1.        ]])

In [39]:
y

0          0
1          1
2          0
3          0
4          0
          ..
1860326    1
1860327    0
1860328    1
1860329    0
1860330    1
Name: target, Length: 1860331, dtype: int64

In [25]:
import joblib

joblib.dump(X_processed, "C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/X_processed.pkl")
joblib.dump(y, "C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y.pkl")


['C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y.pkl']

In [48]:
X_processed = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/X_processed.pkl")

y = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y.pkl")

In [49]:
X_processed

array([[-1.06887082, -1.06850681, -1.06916363, ...,  1.        ,
         0.        ,  1.        ],
       [-1.34756305, -1.34728134, -1.34510439, ...,  1.        ,
         0.        ,  1.        ],
       [-1.35871074, -1.35843232, -1.35625351, ...,  1.        ,
         0.        ,  1.        ],
       ...,
       [-0.50591252, -0.50538224, -0.50334571, ...,  1.        ,
         0.        ,  1.        ],
       [-0.95739393, -0.95699699, -0.95488513, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.71805147,  1.71923856,  1.72090404, ...,  1.        ,
         0.        ,  1.        ]])